# Experiment 4: clustering-objective landscape

This notebook will coordinate the Sobol search, metric-vector calculation, objective scoring, landscape visualization, and local refinement. Reusable logic belongs in `src/cross_species/`.

In [ ]:
from pathlib import Path
import sys

def find_pipeline_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "Cross Species Pipeline", cwd.parent, cwd.parent.parent]
    for candidate in candidates:
        if (candidate / "src" / "cross_species").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate Cross Species Pipeline/src/cross_species")

PIPELINE_ROOT = find_pipeline_root()
PROJECT_ROOT = PIPELINE_ROOT.parent
SRC_ROOT = PIPELINE_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
RESULTS_ROOT = PIPELINE_ROOT / "results"
DATA_ROOT = PROJECT_ROOT / "data"
print(f"Pipeline root: {PIPELINE_ROOT}")

In [ ]:
import pandas as pd

from cross_species import (
    educated_full_score_spec, fit_first_pass_score_spec, load_core_run,
    metric_metadata, metric_vector,
    score_objective, score_objective_table,
)
from cross_species.metrics import METRIC_REGISTRY

MEANINGFUL_MODULE_SIZE = 10
METRIC_NAMES = list(METRIC_REGISTRY)
metric_spec = metric_metadata()
print(f"Objective vector contains {len(METRIC_NAMES)} named metrics.")
display(metric_spec.groupby("category").size().rename("n_metrics"))
display(metric_spec)

## 1. Load one representation

Select one core run and one fixed second-order matrix. Separate searches should be run for no-PCA and each PCA representation.

In [ ]:
CORE_RUN_DIR = None  # Set explicitly, or use the newest results/core/core_* run.

if CORE_RUN_DIR is None:
    candidates = [path for path in (RESULTS_ROOT / "core").glob("core_*") if path.is_dir()]
    if not candidates:
        raise FileNotFoundError("No core run found. Run 01_full_pipeline.ipynb first.")
    CORE_RUN_DIR = max(candidates, key=lambda path: path.stat().st_mtime)

core = load_core_run(CORE_RUN_DIR)
prof = core["prof"]
C = core["C"]
display(pd.Series(core["manifest"], name="value"))

## 2. Define the search space

The first implementation will define ranges for `k_intra`, `k_inter`, `cutoff_intra`, `cutoff_inter`, and Leiden resolution. The Sobol design and duplicate-setting handling will live in `optimization.py`.

## 3. Evaluate each clustering

For every setting, construct the graph, run Leiden, call `evaluate_joint_modules` once, and then call:

```python
y = metric_vector(
    evaluation,
    min_module_size=MEANINGFUL_MODULE_SIZE,
)
```

The evaluation and 65-coordinate metric row should be saved immediately so the search can resume without retaining every graph in memory. The five quantiles belonging to one underlying distribution should later be combined as a group before category scoring; otherwise that property would receive excess weight simply because it has more coordinates.

## 4. Define and freeze the scoring specification

Create the provisional fixed specification, inspect all 65 rules, and reuse it unchanged for every candidate:

```python
score_spec = educated_full_score_spec()
display(score_spec.rules_frame())

scored_runs = score_objective_table(calibration_metrics, score_spec)
display(scored_runs.sort_values("final_score", ascending=False))

# For one new clustering:
score = score_objective(y, score_spec)
print(score.final_score)
display(score.category_scores)
display(score.family_scores)
display(score.desirabilities)
```

The current thresholds are broad educated estimates. Review which desirabilities saturate at 0 or 1 after the initial Sobol run, revise scientifically implausible targets, and then freeze the final specification before local refinement. `fit_first_pass_score_spec(calibration_metrics)` remains available for a separate empirical sensitivity analysis.

## 5. Inspect the global landscape and refine regions

Planned outputs include Pareto plots, parameter-response plots, constraint diagnostics, surrogate validation, and local slices around promising regions.